# Tunisia solar priority zones: six clustering models compared

This notebook starts from the prepared table (1,659 grid cells of 10 km covering Tunisia) and compares six clustering models on seven features: solar output, population density, built-up, cropland and bare-land shares, and distances to a main road and to a power line.

The full pipeline that builds this table from raw open data is on GitHub (link in the dataset description). This is a screening exercise, not a siting decision.

In [ ]:
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch, SpectralClustering, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

# Works on Kaggle (dataset attached to the notebook) and locally (file next to the notebook).
path = (glob.glob('/kaggle/input/**/features_clean.csv', recursive=True) + glob.glob('**/features_clean.csv', recursive=True))[0]
df = pd.read_csv(path)
print(df.shape)
df.head()

## How many clusters?

In [ ]:
FEATURES = ['PVOUT', 'log_pop_density', 'pct_builtup', 'pct_cropland', 'pct_bare', 'log_dist_road_km', 'log_dist_power_line_km']
df = df.dropna(subset=FEATURES).reset_index(drop=True)
X = StandardScaler().fit_transform(df[FEATURES])

rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(X, km.labels_)})
scores = pd.DataFrame(rows); display(scores.round(3))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(scores.k, scores.inertia, 'o-'); ax[0].set_title('Elbow (inertia)')
ax[1].plot(scores.k, scores.silhouette, 'o-'); ax[1].set_title('Silhouette'); plt.show()

## Six models on the same features

There is no ground truth, so the scores describe how compact and separated the groups are, and the ARI column shows how much each model agrees with K-Means (1 = identical).

In [ ]:
K = 3
labels = {'K-Means': KMeans(n_clusters=K, n_init=50, random_state=42).fit_predict(X),
          'Agglomerative (Ward)': AgglomerativeClustering(n_clusters=K, linkage='ward').fit_predict(X),
          'Gaussian Mixture': GaussianMixture(n_components=K, covariance_type='full', n_init=5, random_state=42).fit_predict(X),
          'Birch': Birch(n_clusters=K).fit_predict(X),
          'Spectral': SpectralClustering(n_clusters=K, affinity='nearest_neighbors', n_neighbors=15, assign_labels='kmeans', random_state=42).fit_predict(X)}
for mcs in (max(15, len(X) // 50), 15, 8):          # HDBSCAN chooses its own number of clusters; -1 means noise
    h = HDBSCAN(min_cluster_size=mcs).fit_predict(X)
    if len(set(h) - {-1}) >= 2:
        labels['HDBSCAN'] = h; break

base = labels['K-Means']; out = []
for name, lab in labels.items():
    ok = lab != -1
    out.append({'model': name, 'clusters': len(set(lab[ok])), 'noise_%': round(100 * (~ok).mean(), 1),
                'silhouette': silhouette_score(X[ok], lab[ok]), 'davies_bouldin': davies_bouldin_score(X[ok], lab[ok]),
                'calinski_harabasz': calinski_harabasz_score(X[ok], lab[ok]), 'ARI_vs_KMeans': adjusted_rand_score(base, lab)})
display(pd.DataFrame(out).set_index('model').round(3))

## What the K-Means clusters mean

In [ ]:
df['cluster'] = base
profile = df.groupby('cluster')[['PVOUT', 'pop_density', 'pct_builtup', 'pct_cropland', 'pct_bare', 'dist_road_km', 'dist_power_line_km']].mean().round(1)
profile.insert(0, 'cells', df.groupby('cluster').size()); display(profile)

# Each point is one 10 km cell placed at its centre, so the scatter reads as a map of Tunisia.
n = len(labels); fig, axes = plt.subplots(2, 3, figsize=(13, 13)); axes = axes.ravel()
for ax, (name, lab) in zip(axes, labels.items()):
    ax.scatter(df.lon, df.lat, c=lab, cmap='tab10', s=9, marker='s'); ax.set_title(name); ax.set_aspect(1.2); ax.set_axis_off()
for ax in axes[n:]: ax.set_axis_off()
plt.tight_layout(); plt.show()

**Data credits:** Global Solar Atlas (World Bank Group, ESMAP, Solargis, CC BY 4.0); WorldPop; ESA WorldCover 2021 v200; © OpenStreetMap contributors (ODbL); geoBoundaries.